# FRED ingestion - worker

Downloads one macroeconomic series from the FRED API, writes it to `finhive.fred.<series>`, and exits with a JSON payload the orchestrator appends to `finhive.logs.ingestionLog`. The FRED API key is read from the `finhive` secret scope.

In [ ]:
dbutils.widgets.text("series", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("job_name", "")

series = dbutils.widgets.get("series")
start_date = dbutils.widgets.get("start_date") or None
job_name = dbutils.widgets.get("job_name")

if not series:
    raise ValueError("widget 'series' is required")

In [ ]:
import json
from datetime import datetime, timezone

from fredapi import Fred
from pyspark.sql import functions as F

SOURCE = "Fred"

try:
    fred_api_key = dbutils.secrets.get(scope="finhive", key="fred-api-key")
    fred = Fred(api_key=fred_api_key)

    observations = fred.get_series(series, observation_start=start_date)
    if observations.empty:
        raise ValueError(f"FRED returned no observations for series '{series}'")

    pdf = observations.rename("value").rename_axis("date").reset_index()
    last_observation_date = pdf["date"].max().strftime("%Y-%m-%d")

    ingested_at = datetime.now(timezone.utc)
    sdf = (
        spark.createDataFrame(pdf)
        .withColumn("ingested_at", F.lit(ingested_at))
        .withColumn("pipeline_name", F.lit(job_name))
    )

    spark.sql("CREATE CATALOG IF NOT EXISTS finhive")
    spark.sql("CREATE SCHEMA IF NOT EXISTS finhive.fred")

    table_name = f"finhive.fred.`{series}`"
    sdf.write.mode("append").saveAsTable(table_name)

    result = {
        "series": series,
        "source": SOURCE,
        "status": True,
        "updateAt": ingested_at.isoformat(),
        "lastObservationDate": last_observation_date,
        "item_count": sdf.count(),
        "error": None,
        "job_name": job_name,
    }
except Exception as e:
    result = {
        "series": series,
        "source": SOURCE,
        "status": False,
        "updateAt": datetime.now(timezone.utc).isoformat(),
        "lastObservationDate": None,
        "item_count": 0,
        "error": str(e),
        "job_name": job_name,
    }

dbutils.notebook.exit(json.dumps(result))